# FlashSpec Colab + Google Drive 运行脚本

运行前先在 Colab 菜单选择 `Runtime -> Change runtime type -> GPU`，如果要验证 A100，硬件加速器选择 A100。

这个 notebook 会把代码仓库放在 Google Drive：`/content/drive/MyDrive/FlashSpecColab/repo`，benchmark JSON 结果保存在：`/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/`。这样 Colab runtime 重启后，代码和结果仍然保留在云盘里。

## 0. 挂载 Google Drive

第一次运行会弹出授权。授权后，Colab 可以读写你的 Google Drive。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/FlashSpecColab")
PROJECT_DIR = DRIVE_ROOT / "repo"
RESULTS_DIR = DRIVE_ROOT / "results" / "colab_kernels"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## 1. 检查 Colab GPU 环境

如果这里报错，说明当前 runtime 不是 GPU，需要先切换 Colab runtime。

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("当前 Colab runtime 没有 CUDA GPU，请先切换到 GPU/A100 runtime。")

## 2. 在 Google Drive 中拉取代码并安装依赖

如果云盘里已经有 `/content/drive/MyDrive/FlashSpecColab/repo/.git`，就执行 `git pull --ff-only`；否则 clone 到云盘。`.[triton]` 会安装 Triton 可选依赖，用于运行真实 Kernel 1。

In [ ]:
import os
import subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/FlashSpecColab")
PROJECT_DIR = DRIVE_ROOT / "repo"
RESULTS_DIR = DRIVE_ROOT / "results" / "colab_kernels"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/honey-floria/FlashSpec.git", str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(
        f"{PROJECT_DIR} 已存在但不是 Git 仓库。请删除/重命名这个目录，"
        "或者把 PROJECT_DIR 改成一个新的空目录。"
    )

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

%pip install -e ".[triton]"

## 3. 检查 FlashSpec、CUDA 和 Triton 状态

`HAS_TRITON=True` 且 `cuda available=True` 时，`triton_fused` benchmark 会走真实 Triton fused dequant attention kernel。

In [ ]:
import sys
import torch
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/FlashSpecColab/repo")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from flashspec.triton_kernels import HAS_TRITON

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("HAS_TRITON:", HAS_TRITON)

## 4. 运行正确性测试

这里会包含 CUDA + Triton 可用时的 Kernel 1 correctness 测试：输出对齐 PyTorch reference，并检查 `materializes_dense_kv == 0.0`。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab/repo
!python -m unittest discover -s tests

## 5. Kernel 1 microbenchmark 对比

- `dense`: dense KV reference attention。
- `fused`: portable PyTorch INT8 KV 路径，会先 materialize dense KV。
- `triton_fused`: 真实 Kernel 1，直接读取 INT8 K/V，在 Triton kernel 内完成反量化、QK、softmax 和 PV。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab/repo

# 小规模 smoke benchmark，先确认三条路径都能跑通。
!python benchmarks/microbench.py --backend dense --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend fused --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend triton_fused --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend paged --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --device cuda --dtype float16 --json
!python benchmarks/microbench.py --backend triton_paged --batch 4 --heads 8 --seq-len 512 --head-dim 64 --block-size 16 --iters 10 --device cuda --dtype float16 --json

## 6. A100 目标 shape benchmark

TODO P0.1/P0.2 要覆盖 `head_dim=64/128` 和 `seq_len=512/2048/4096`。下面跑 Kernel 1 Triton fused 和 Kernel 2 Triton paged 主路径，并把 JSON 保存到 Google Drive 的 `/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/`。如果显存或 Colab 时间不够，可以减少 `--batch` 或 `--iters`。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab/repo
!mkdir -p /content/drive/MyDrive/FlashSpecColab/results/colab_kernels

!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 512 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s512_d64.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 2048 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s2048_d64.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 4096 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s4096_d64.json

!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 512 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s512_d128.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s2048_d128.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 4096 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s4096_d128.json

!python benchmarks/microbench.py --backend triton_paged --batch 16 --heads 32 --seq-len 512 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s512_d64.json
!python benchmarks/microbench.py --backend triton_paged --batch 16 --heads 32 --seq-len 2048 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s2048_d64.json
!python benchmarks/microbench.py --backend triton_paged --batch 16 --heads 32 --seq-len 4096 --head-dim 64 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s4096_d64.json

!python benchmarks/microbench.py --backend triton_paged --batch 16 --heads 32 --seq-len 512 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s512_d128.json
!python benchmarks/microbench.py --backend triton_paged --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s2048_d128.json
!python benchmarks/microbench.py --backend triton_paged --batch 16 --heads 32 --seq-len 4096 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s4096_d128.json

## 7. dense / portable fused / Triton fused 保存对比

这组命令用于同一个 shape 下对比三条路径。重点看 `latency_ms`、`tokens_per_second`、`compression_ratio` 和 `materializes_dense_kv`。所有 JSON 都会保存到 Google Drive。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab/repo
!mkdir -p /content/drive/MyDrive/FlashSpecColab/results/colab_kernels

!python benchmarks/microbench.py --backend dense --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/dense_s2048_d128.json
!python benchmarks/microbench.py --backend fused --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/portable_fused_s2048_d128.json
!python benchmarks/microbench.py --backend triton_fused --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_fused_s2048_d128_compare.json
!python benchmarks/microbench.py --backend paged --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/portable_paged_s2048_d128.json
!python benchmarks/microbench.py --backend triton_paged --batch 16 --heads 32 --seq-len 2048 --head-dim 128 --block-size 16 --iters 50 --device cuda --dtype float16 --json --output /content/drive/MyDrive/FlashSpecColab/results/colab_kernels/triton_paged_s2048_d128_compare.json

## 8. 汇总云盘中的结果

读取 Google Drive 上的 `/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/*.json`，快速查看每个 benchmark 的关键字段。

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("/content/drive/MyDrive/FlashSpecColab/results/colab_kernels")

for path in sorted(RESULTS_DIR.glob("*.json")):
    data = json.loads(path.read_text())
    print(
        path.name,
        "backend=", data.get("backend"),
        "latency_ms=", round(float(data.get("latency_ms", 0.0)), 4),
        "tok/s=", round(float(data.get("tokens_per_second", 0.0)), 2),
        "compression=", round(float(data.get("compression_ratio", 0.0)), 3),
        "materializes_dense_kv=", data.get("materializes_dense_kv"),
    )